In [1]:
import pandas as pd
import ast
import wandb.apis.public
import itertools
import concurrent.futures
import tqdm
from pathlib import Path
import json
import os
import sys
sys.path.append("..")

import wandb
os.environ["WANDB_SILENT"] = "true" # Do not print download output.
api = wandb.Api(timeout=100)

In [2]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [3]:
project_id = "qnrl"

In [4]:
# List of sweep names and their IDs for reference.

# # OLD sweeps that should be deleted.
# sweep_name_id = { # name: id
#     'qnrl/FrozenLake-v1': 'oi7rm7sh',
#     'qnrl/CliffWalking-v1': '2f7ji0uu',
#     'qnrl/Acrobot-v1': '28tqdi2r',
#     'qnrl/CartPole-v1': '0xhzu1tq',
#     'c51/FrozenLake-v1': '6ebrrbwp',
#     'c51/CliffWalking-v1': '0nxxo52o',
#     'c51/Acrobot-v1': '2ti1laob',
#     'c51/CartPole-v1': 'mjqmhnd0',
# }

# NEW.
sweep_name_id = { # name: id
    # QnRL.
    'qnrl/FrozenLake-v1': 'sgi6zb0t',
    'qnrl/CliffWalking-v1': 'h4g5er22',
    'qnrl/Acrobot-v1': 'sd4va92r',
    'qnrl/CartPole-v1': '4ykq4hvw',
    
    # C51.
    'c51/FrozenLake-v1': '3exa9jvo',
    'c51/CliffWalking-v1': '1nlfftsy',
    'c51/Acrobot-v1': 's3xwn85k',
    'c51/CartPole-v1': '7raha9om',
}

In [5]:
# # Get existence status of history artifact for all runs.
# history_exists = [
#     (sweep_id, run.id, api.artifact_exists(name=f"{project_id}/run-{run.id}-history:latest", type="wandb-history"))
#     for sweep_id in sweep_name_id.values() for run in api.sweep(f"{project_id}/{sweep_id}").runs
# ]

In [6]:
# # Report which runs (if any) do not have a history artifact yet.
# list(filter(lambda tup: tup[-1] == False, history_exists))

In [7]:
# sizes = [
#     (sweep_id, run_id, api.artifact(name=f"{project_id}/run-{run_id}-history:latest", type="wandb-history").size)
#     for sweep_id, run_id, exists in history_exists
#     if exists
# ]

In [8]:
# sizes[:3]

In [9]:
def get_run_config_dict(run: wandb.apis.public.Run):
    """Compute run config hash from original run program command string."""
    # Download the run metadata file, load as JSON dictionary, and then delete the local downloaded file.
    metadata_file = run.file("wandb-metadata.json").download(replace=True)
    metadata = json.load(metadata_file)
    if os.path.exists(metadata_file.name):
        os.remove(metadata_file.name)
    
    # Get config string from run program command arguments.
    # Format is: --config="{...}"
    cfg_str = metadata['args'][1].split("--config=")[1]
    # Convert this string to dictionary via literal evaluation.
    cfg_dict = ast.literal_eval(cfg_str)
    return cfg_dict

In [10]:
import shutil
from typing import Literal
import numpy as np


def download_run_data(
    root: Path,
    run: wandb.apis.public.Run,
    replace_existing: bool = False,
    skip_cache: bool = False,
    history_source: Literal['artifact', 'scan'] = 'artifact',
    scan_page_size: int = 100000,
    ):
    """Download files for the given run. Returns a list of functions that when called will download the associated file."""
    root_run = root / f"run-{str(run.id)}"
    root_run.mkdir(parents=True, exist_ok=True)
    
    # TODO download all files for a run.
    # list(api.run("zanderman-virginia-tech/qnrl/o8bvmgah").files())
    
    def _download_config():
        """Download run config."""
        run.file("config.yaml").download(root=str(root_run), replace=replace_existing, exist_ok=True)
    
    def _download_summary():
        """Download run summary."""
        run.file("wandb-summary.json").download(root=str(root_run), replace=replace_existing, exist_ok=True)
    
    def _download_metadata():
        """Download run metadata."""
        run.file("wandb-metadata.json").download(root=str(root_run), replace=replace_existing, exist_ok=True)
    
    def _download_history():
        """Download run metrics history."""
        root_history = root_run / "history"
        artifact_kwargs = dict(name=f"{project_id}/run-{run.id}-history:latest", type="wandb-history")
        
        # See also new `run.download_history_exports()` from v0.24.1+
        if history_source == 'artifact' and api.artifact_exists(**artifact_kwargs):
            history_artifact = api.artifact(**artifact_kwargs)
            if not root_history.exists() or replace_existing:
                # Remove existing.
                if root_history.exists(): shutil.rmtree(root_history)
                # Create new directory.
                root_history.mkdir(parents=True, exist_ok=True)
                # Download artifact.
                history_artifact.download(root=str(root_history), skip_cache=skip_cache)
        else:
            outfile = root_history / "0000.parquet"
            if not outfile.exists() or replace_existing:
                print(f"[{run.id}] download via scan_history")
                df = pd.DataFrame(list(run.scan_history(page_size=scan_page_size)))
                # Replace known strings with equivalents.
                df = (df
                    .replace(['Infinity', '+Infinity'], np.inf)
                    .replace('-Infinity', -np.inf)
                    .replace('NaN', np.nan)
                    .infer_objects(copy=False) # Infer column type from objects within it.
                )
                
                # Remove existing.
                if root_history.exists(): shutil.rmtree(root_history)
                # Create new directory.
                root_history.mkdir(parents=True, exist_ok=True)
                # Save artifact to file.
                df.to_parquet(outfile)
        
    def _download_models():
        """Download run saved models."""
        artifact_kwargs = dict(name=f"{project_id}/run-{run.id}-models:latest", type="model")
        if api.artifact_exists(**artifact_kwargs):
            models_artifact = api.artifact(**artifact_kwargs)
            root_models = root_run / "models"
            if not root_models.exists() or replace_existing:
                # Remove existing.
                if root_models.exists(): shutil.rmtree(root_models)
                # Create new directory.
                root_models.mkdir(parents=True, exist_ok=True)
                # Download artifact.
                models_artifact.download(root=str(root_models), skip_cache=skip_cache)
    
    return [
        _download_config,
        _download_summary,
        _download_metadata,
        _download_history,
        _download_models,
    ]

def download_all_run_data_for_sweep(
    root: Path,
    sweep: wandb.apis.public.Sweep,
    replace_existing: bool = False,
    skip_cache: bool = False,
    history_source: Literal['artifact', 'scan'] = 'artifact',
    scan_page_size: int = 100000,
    ):
    """Download files for all runs associated with the given sweep."""
    root_sweep = root / f"sweep-{str(sweep.id)}"
    root_sweep.mkdir(parents=True, exist_ok=True)
    return list(itertools.chain.from_iterable(
        download_run_data(
            root=root_sweep,
            run=run,
            replace_existing=replace_existing,
            skip_cache=skip_cache,
            history_source=history_source,
            scan_page_size=scan_page_size,
        )
        for run in sweep.runs
    ))

In [11]:
# ###
# # Download a specific run.
# ###
# # runs_to_download = ['ixfifpwm', 'z3k9i6db', 'nujoh0r7', 'm7w1ciph']
# # runs_to_download = ['ixfifpwm', 'z3k9i6db', 'nujoh0r7', 'm7w1ciph']
# runs_to_download = ['ixfifpwm']
# # root_runs = root_sweeps/f"sweep-sd4va92r"

# files_to_download = list(tqdm.tqdm(
#     itertools.chain.from_iterable(
#         download_run_data(
#             root=root_sweeps / f"sweep-{api.run(f'{project_id}/{run_id}').sweep.id}",
#             run=api.run(f"{project_id}/{run_id}"),
#             replace_existing=True,
#         )
#         for run_id in runs_to_download
#     ),
#     unit="files",
#     desc="Preparing downloads",
# ))

# # Download all files and folders for given sweeps.
# num_threads = min(32, (os.cpu_count() or 1) + 4) # Maximum number of concurrent threads
# with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
#     list(tqdm.tqdm(executor.map(lambda f: f(), files_to_download), total=len(files_to_download), unit="files", desc="Downloading"))

In [12]:
replace_existing = False
skip_cache = True
history_source = 'scan'

###
# Download all sweeps.
###
sweeps_to_download = list(sweep_name_id.values())

# Collect list of files and folders to download.
files_to_download = list(tqdm.tqdm(
    itertools.chain.from_iterable(
        download_all_run_data_for_sweep(
            root=root_sweeps, 
            sweep=api.sweep(f"{project_id}/{sweep_id}"),
            replace_existing=replace_existing,
            skip_cache=skip_cache,
            history_source=history_source,
        )
        for sweep_id in sweeps_to_download
    ),
    unit="files",
    desc="Preparing downloads",
))

# Download all files and folders for given sweeps.
num_threads = min(32, (os.cpu_count() or 1) + 4) # Maximum number of concurrent threads
print(f"{num_threads} threads")
with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
    list(tqdm.tqdm(executor.map(lambda f: f(), files_to_download), total=len(files_to_download), unit="files", desc="Downloading"))

Preparing downloads: 2240files [00:43, 51.20files/s]


18 threads


Downloading:  90%|████████▉ | 2013/2240 [03:43<00:16, 14.16files/s]

[gbkzvg60] download via scan_history


Downloading:  90%|█████████ | 2026/2240 [03:44<00:18, 11.35files/s]/var/folders/t7/1jlv9_sn36g96901km63r92m0000gn/T/ipykernel_82789/589356760.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(['Infinity', '+Infinity'], np.inf)
Downloading:  91%|█████████ | 2028/2240 [03:45<00:24,  8.73files/s]

[bgrjs1z8] download via scan_history


... (57 outputs left out)

[jit954em] download via scan_history


Downloading:  99%|█████████▉| 2216/2240 [04:53<00:11,  2.03files/s]

[x0tak5kj] download via scan_history


Downloading: 100%|██████████| 2240/2240 [04:57<00:00,  7.53files/s]
